In [1]:
#%cd ..

/workspace/notebooks/GitProjects/hybrid-recsys


In [ ]:
import torch
from baseline.RecVAE.model import VAE

# Load the full dictionary
checkpoint = torch.load('checkpoints/recVAE/recvae_best_model.pth', weights_only=False)

# Re-create the architecture using the saved dict
# The ** operator unpacks the dictionary directly into the class constructor
model = VAE(**checkpoint['model_kwargs'])
model.load_state_dict(checkpoint['model_state_dict'])

model.eval()

VAE(
  (encoder): Encoder(
    (fc1): Linear(in_features=10581, out_features=600, bias=True)
    (ln1): LayerNorm((600,), eps=0.1, elementwise_affine=True)
    (fc2): Linear(in_features=600, out_features=600, bias=True)
    (ln2): LayerNorm((600,), eps=0.1, elementwise_affine=True)
    (fc3): Linear(in_features=600, out_features=600, bias=True)
    (ln3): LayerNorm((600,), eps=0.1, elementwise_affine=True)
    (fc4): Linear(in_features=600, out_features=600, bias=True)
    (ln4): LayerNorm((600,), eps=0.1, elementwise_affine=True)
    (fc5): Linear(in_features=600, out_features=600, bias=True)
    (ln5): LayerNorm((600,), eps=0.1, elementwise_affine=True)
    (fc_mu): Linear(in_features=600, out_features=200, bias=True)
    (fc_logvar): Linear(in_features=600, out_features=200, bias=True)
  )
  (prior): CompositePrior(
    (encoder_old): Encoder(
      (fc1): Linear(in_features=10581, out_features=600, bias=True)
      (ln1): LayerNorm((600,), eps=0.1, elementwise_affine=True)
      (f

In [ ]:
def get_recommendations(model, user_liked_ids, sid2idx, unique_sids, top_k=10):
    model.eval()
    num_items = len(unique_sids)
    
    # Map Raw IDs to Indices (Handling OOV)
    liked_indices = [sid2idx[sid] for sid in user_liked_ids if sid in sid2idx]
    
    # Create Multi-Hot Vector
    input_vector = torch.zeros(1, num_items)
    input_vector[0, liked_indices] = 1.0
    
    # Inference 
    with torch.no_grad():
        # Passing calculate_loss=False to get the decoder output
        logits = model(input_vector, calculate_loss=False)

    # Move to CPU/Numpy and flatten
    scores = logits.cpu().numpy().squeeze()
    
    # Mask already seen items (set to very low value)
    scores[liked_indices] = -1e9
    
    # Rank and Map back to Anime IDs
    top_indices = np.argsort(scores)[::-1][:top_k]
    recommended_ids = [unique_sids[i] for i in top_indices]
    
    return recommended_ids

In [23]:
import pandas as pd

# Load only the columns we care about
meta_df = pd.read_csv('data/raw/mal/anime-dataset-2023.csv', usecols=['anime_id', 'Name', 'English name'])

# Create a dictionary for O(1) title lookup
# We'll use 'Name' as the default, but you could swap for 'English name'
id_to_name = meta_df.set_index('anime_id')['Name'].to_dict()

In [7]:
import numpy as np

# Load the file (assuming it's a single column of IDs)
unique_sids = np.loadtxt('data/processed/recVAE/unique_sid.txt', dtype=int)

# Create the inverse mapping: {AnimeID: Index}
sid2idx = {sid: i for i, sid in enumerate(unique_sids)}

# Total number of items the model knows
num_items = len(unique_sids)

In [ ]:
user_likes = [5114, 11061, 99999] # Example raw Anime IDs

# Map to indices, ignoring IDs the model never saw during training
input_indices = [sid2idx[sid] for sid in user_likes if sid in sid2idx]

# Initialize a vector of zeros the size of your total items
input_vector = torch.zeros(num_items)

# construct the multi-hot vector
input_vector[input_indices] = 1.0

# RecVAE expects a batch dimension (Batch Size, Number of Items)
input_tensor = input_vector.unsqueeze(0)
with torch.no_grad():
    preds = model(input_tensor, calculate_loss=False)
preds[0]

tensor([ 7.8100, -0.3046, -0.5731,  ..., -7.5002, -6.4055, -5.8918])

In [ ]:
def recommend_with_titles(model, user_liked_ids, sid2idx, unique_sids, id_to_name_map, top_k=10):
    # Get the raw Anime IDs using the function from before
    rec_ids = get_recommendations(model, user_liked_ids, sid2idx, unique_sids, top_k)
    
    print(f"--- Recommendations for User ---")
    results = []
    for i, aid in enumerate(rec_ids, 1):
        # Lookup name; default to "Unknown ID: [id]" if not found
        title = id_to_name_map.get(aid, f"Unknown ID: {aid}")
        results.append(title)
        print(f"{i}. {title}")
        
    return results

# Example Usage:
user_history = [32379] 
titles = recommend_with_titles(model, user_history, sid2idx, unique_sids, id_to_name)

--- Recommendations for User ---
1. Berserk 2nd Season
2. Kenpuu Denki Berserk
3. Berserk: Ougon Jidai-hen III - Kourin
4. Berserk: Ougon Jidai-hen II - Doldrey Kouryaku
5. Berserk: Ougon Jidai-hen I - Haou no Tamago
6. Bleach
7. Death Note
8. One Piece
9. Cowboy Bebop
10. Claymore
